In [ ]:
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

from openai import OpenAI

client = OpenAI(api_key="")

c:\Users\vinee\Desktop\AI research Engineer SHL\3MarEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import json
import pandas as pd

# structured dataset
with open("processed_assessments_clean.json", "r", encoding="utf-8") as f:
    structured = json.load(f)

df_structured = pd.DataFrame(structured)

# original catalog dataset
with open("assessments_catalog.json", "r", encoding="utf-8") as f:
    catalog = json.load(f)

df_catalog = pd.DataFrame(catalog)

print(len(df_structured), len(df_catalog))

377 377


In [5]:
df = df_structured.merge(
    df_catalog[["url","description"]],
    on="url",
    how="left"
)

df["description"] = df["description"].fillna("")

In [6]:
def build_structured_text(row):

    parts = []

    if row.get("assessed_skills_norm"):
        parts.append("Skills: " + ", ".join(row["assessed_skills_norm"]))

    if row.get("target_roles_norm"):
        parts.append("Roles: " + ", ".join(row["target_roles_norm"]))

    if row.get("cognitive_dimensions_norm"):
        parts.append("Cognitive: " + ", ".join(row["cognitive_dimensions_norm"]))

    return " ".join(parts)

df["structured_text"] = df.apply(build_structured_text, axis=1)

# Approach texts
df["text_a1"] = df["description"] + " " + df["structured_text"]
df["text_a2"] = df["description"]
df["text_a3"] = df["structured_text"]

In [7]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

c:\Users\vinee\Desktop\AI research Engineer SHL\3MarEnv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
def compute_embeddings(texts):

    emb = model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return emb

emb_a1 = compute_embeddings(df["text_a1"].tolist())
emb_a2 = compute_embeddings(df["text_a2"].tolist())
emb_a3 = compute_embeddings(df["text_a3"].tolist())

Batches: 100%|██████████| 12/12 [00:07<00:00,  1.68it/s]


In [9]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_features=50000
)

tfidf_matrix = tfidf.fit_transform(df["text_a1"])
tfidf_matrix = normalize(tfidf_matrix)

In [10]:
def parse_query(query):

    prompt = f"""
You are an expert HR assessment consultant.

Your task is to extract structured hiring requirements from the query.

Return JSON with these fields:

looking_for:
Describe the role, skills, and competencies being assessed.

constraints:
Include time limits, experience requirements, or other restrictions.

job_level:
Infer seniority level if possible (entry, mid, senior, executive).

skills:
List explicit technical or behavioral skills mentioned.

language:
Required test language if specified.

Query:
{query}

Return JSON only.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    txt = response.choices[0].message.content

    m = re.search(r"\{.*\}", txt, re.DOTALL)

    if m:
        return json.loads(m.group())

    return {}

In [11]:
def expand_query(parsed):

    text = parsed.get("looking_for","")

    skills = parsed.get("skills",[])

    if isinstance(skills,list):
        text += " " + " ".join(skills)

    return text

In [12]:
def retrieve_candidates(query_text, embeddings, top_k=100):

    q_emb = model.encode(query_text, normalize_embeddings=True)

    sims = cosine_similarity([q_emb], embeddings)[0]

    idx = np.argsort(sims)[::-1][:top_k]

    return idx, sims

In [13]:
def hybrid_score(query_text, idx, sims):

    q_tfidf = tfidf.transform([query_text])
    tfidf_scores = (tfidf_matrix[idx] @ q_tfidf.T).toarray().ravel()

    scores = 0.7*sims[idx] + 0.3*tfidf_scores

    return scores

In [14]:
def recommend(query, embeddings):

    parsed = parse_query(query)

    expanded = expand_query(parsed)

    idx, sims = retrieve_candidates(expanded, embeddings)

    scores = hybrid_score(expanded, idx, sims)

    res = df.iloc[idx].copy()
    res["score"] = scores

    res = res.sort_values("score",ascending=False)

    return res.head(10)

In [15]:
def recall_at_10(pred, truth):

    p = set(pred)
    t = set(truth)

    return len(p & t) / len(t)

In [16]:
eval_df = pd.read_excel("Gen_AI Dataset.xlsx", sheet_name="Train-Set")

grouped = eval_df.groupby("Query")["Assessment_url"].apply(list).to_dict()

In [17]:
def extract_slug(url):
    
    if not isinstance(url,str):
        return ""
    
    url = url.lower().strip()
    
    # remove protocol
    url = url.replace("https://","").replace("http://","")
    
    # remove query/fragment
    url = re.sub(r"[?#].*$","",url)
    
    # extract slug
    m = re.search(r"/view/([^/]+)",url)
    
    if m:
        return m.group(1)
    
    # fallback
    parts = url.split("/")
    return parts[-1] if parts else url

In [18]:
def recall_at_10(pred_urls, true_urls):

    pred_slugs = {extract_slug(u) for u in pred_urls}
    true_slugs = {extract_slug(u) for u in true_urls}

    if len(true_slugs) == 0:
        return 0

    hits = len(pred_slugs & true_slugs)

    return hits / len(true_slugs)

In [19]:
def evaluate(embeddings):

    recalls = []

    for q, urls in tqdm(grouped.items()):

        res = recommend(q, embeddings)

        preds = res["url"].tolist()

        r = recall_at_10(preds, urls)

        recalls.append(r)

    return np.mean(recalls)

In [20]:
recall_a1 = evaluate(emb_a1)
recall_a2 = evaluate(emb_a2)
recall_a3 = evaluate(emb_a3)

print("Approach 1 (description + structured):", round(recall_a1,4))
print("Approach 2 (description only):", round(recall_a2,4))
print("Approach 3 (structured only):", round(recall_a3,4))

100%|██████████| 10/10 [00:28<00:00,  2.83s/it]

Approach 1 (description + structured): 0.2767
Approach 2 (description only): 0.2033
Approach 3 (structured only): 0.2911


In [21]:
import pandas as pd
from tqdm import tqdm

rows = []

for query, true_urls in tqdm(grouped.items()):

    res = recommend(query, emb_a1)

    pred_urls = res["url"].tolist()
    pred_names = res["name"].tolist()
    scores = res["score"].tolist()

    true_slugs = {extract_slug(u) for u in true_urls}

    for rank, (name, url, score) in enumerate(zip(pred_names, pred_urls, scores), start=1):

        slug = extract_slug(url)

        rows.append({
            "query": query,
            "rank": rank,
            "assessment_name": name,
            "predicted_url": url,
            "score": score,
            "predicted_slug": slug,
            "is_relevant": slug in true_slugs,
            "ground_truth_urls": ", ".join(true_urls)
        })

df_results = pd.DataFrame(rows)

df_results.to_excel("approach1_recommendations.xlsx", index=False)

print("Saved to: approach1_recommendations.xlsx")

100%|██████████| 10/10 [00:23<00:00,  2.30s/it]

Saved to: approach1_recommendations.xlsx


In [22]:
import json
import re

def debug_parse_query(query):

    prompt = f"""
You are an expert HR assessment consultant.

Your task is to extract structured hiring requirements from the query.

Return JSON with these fields:

looking_for:
Describe the role, skills, and competencies being assessed.

constraints:
Include time limits, experience requirements, or other restrictions.

job_level:
Infer seniority level if possible (entry, mid, senior, executive).

skills:
List explicit technical or behavioral skills mentioned.

language:
Required test language if specified.

Query:
{query}

Return JSON only.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    raw_output = response.choices[0].message.content

    print("\n===== RAW LLM OUTPUT =====\n")
    print(raw_output)

    m = re.search(r"\{.*\}", raw_output, re.DOTALL)

    if m:
        parsed = json.loads(m.group())

        print("\n===== PARSED JSON =====\n")
        print(json.dumps(parsed, indent=2))

        return parsed

    print("\n⚠️ Could not parse JSON")
    return {}

In [26]:
query = """I am looking for a COO for my company in China and I want to see if they are culturally a right fit for our company. Suggest me an assessment that they can complete in about an hour
"""

debug_parse_query(query)


===== RAW LLM OUTPUT =====

```json
{
  "looking_for": "Chief Operating Officer (COO) role with a focus on cultural fit within the company.",
  "constraints": "Assessment should be completed in about an hour.",
  "job_level": "executive",
  "skills": [],
  "language": "not specified"
}
```

===== PARSED JSON =====

{
  "looking_for": "Chief Operating Officer (COO) role with a focus on cultural fit within the company.",
  "constraints": "Assessment should be completed in about an hour.",
  "job_level": "executive",
  "skills": [],
  "language": "not specified"
}


{'looking_for': 'Chief Operating Officer (COO) role with a focus on cultural fit within the company.',
 'constraints': 'Assessment should be completed in about an hour.',
 'job_level': 'executive',
 'skills': [],
 'language': 'not specified'}

In [27]:
import pandas as pd
from tqdm import tqdm

rows = []

for query, true_urls in tqdm(grouped.items()):

    # Run recommendation using Approach 3 embeddings
    res = recommend(query, emb_a3)

    pred_urls = res["url"].tolist()
    pred_names = res["name"].tolist()
    scores = res["score"].tolist()

    true_slugs = {extract_slug(u) for u in true_urls}

    for rank, (name, url, score) in enumerate(zip(pred_names, pred_urls, scores), start=1):

        slug = extract_slug(url)

        rows.append({
            "query": query,
            "rank": rank,
            "assessment_name": name,
            "predicted_url": url,
            "predicted_slug": slug,
            "score": score,
            "is_relevant": slug in true_slugs,
            "ground_truth_urls": ", ".join(true_urls)
        })

df_results = pd.DataFrame(rows)

df_results.to_excel("approach3_recommendations_vs_groundtruth.xlsx", index=False)

print("Saved results to: approach3_recommendations_vs_groundtruth.xlsx")

100%|██████████| 10/10 [00:32<00:00,  3.21s/it]


Saved results to: approach3_recommendations_vs_groundtruth.xlsx
